# core

> An async Jupyter kernel client over HTTP and websockets

In [ ]:
#| default_exp core

jupyasyncclient talks to any server exposing the standard Jupyter kernels API: [jupygate](https://github.com/AnswerDotAI/jupygate), or jupyter_server itself. Kernel lifecycle goes over HTTP (`POST /api/kernels`, interrupt, restart, delete); messaging goes over one websocket carrying the legacy Jupyter protocol (message dicts plus a `channel` key). There is no zmq anywhere in this package, and sends are genuinely awaited - the zmq subtleties (sync-send edge consumption, slow-joiner subscriptions, identity contracts) all live server-side.

The API mirrors jupyter_client's `AsyncKernelClient` where that helps a reader coming from it (`execute`, `kernel_info`, `wait_for_ready`, per-channel `get_*_msg`), with two additions: any request can await its reply directly with `reply=True`, and any `*_request` message type in the protocol is available as a generated method.

Throughout this page the live server is a jupygate running an [ipymini](https://github.com/AnswerDotAI/ipymini) kernel; everything works identically against jupyter_server (the test suite runs against that).

In [ ]:
#| export
import asyncio, json, logging, os, time, uuid, httpx, websockets
from contextlib import suppress
from queue import Empty
from urllib.parse import urlencode, urlsplit, urlunsplit
from jupywire.session import (Session, validate_string_dict, dumps, loads,
    serialize_binary_message, deserialize_binary_message)
from jupywire.ops import EvalOps, EvalException, try_eval
from fastcore.basics import patch, patch_to, nested_idx
from fastcore.nbio import msg2out
from fastcore.meta import use_kwargs_dict


In [ ]:
import time
from fastcore.test import test_eq, test_fail
from jupygate.core import create_app, serve

In [ ]:
#| export
log = logging.getLogger('jupyasyncclient')

## Wire codec

The legacy protocol's two encodings: JSON text frames, and a binary framing (count, offset table, JSON, then the raw buffers) for messages that carry `buffers`. The [jupygate docs](https://AnswerDotAI.github.io/jupygate/core.html) walk the byte layout; the codec (`dumps`/`loads`, `serialize_binary_message`/`deserialize_binary_message`) lives in `jupywire.session`, and here it just needs to round-trip, with inbound buffers as `memoryview`s, zero-copy off the frame.

In [ ]:
ses = Session(key=b'demo')
msg = ses.msg('comm_msg', dict(comm_id='c', data={}))
msg['channel'] = 'iopub'
msg['buffers'] = [b'raw']
back = deserialize_binary_message(serialize_binary_message(msg))
test_eq(bytes(back['buffers'][0]), b'raw')
test_eq(loads(dumps(dict(msg, buffers=[])))['content'], msg['content'])
back['channel']

'iopub'

## Shared HTTP plumbing

The three public classes all speak the same HTTP surface, so it lives in one base: URL joining (including the http-to-ws scheme flip for the channels endpoint), an owned-or-borrowed `httpx.AsyncClient`, token headers, and the `/api/kernels` paths. Subclasses that are bound to one kernel override `kernel_request` to fill in their kernel id.

In [ ]:
#| export
def _join_url(base, path, ws=False, params=None):
    u = urlsplit(base)
    scheme = {"http": "ws", "https": "wss"}.get(u.scheme, u.scheme) if ws else u.scheme
    base_path = u.path.rstrip("/")
    full_path = f"{base_path}/{path.lstrip('/')}" if path else base_path
    query = urlencode({k: v for k, v in (params or {}).items() if v is not None})
    return urlunsplit((scheme, u.netloc, full_path, query, ""))

In [ ]:
#| export
class KernelApi:
    "Shared HTTP plumbing for the Jupyter kernels API."
    def __init__(self, base_url, token=None, headers=None, timeout=30, http_client=None):
        self.base_url,self.token,self._timeout = base_url.rstrip('/'),token or '',timeout
        self._headers = {**(headers or {})}
        if self.token and 'Authorization' not in self._headers: self._headers['Authorization'] = f'token {self.token}'
        self._http,self._own_http = http_client,http_client is None

    def _ensure_http(self):
        if self._http and not self._http.is_closed: return self._http
        self._http,self._own_http = httpx.AsyncClient(headers=self._headers),True
        return self._http

    async def _request(self, method, path, **kwargs):
        r = await self._ensure_http().request(method, _join_url(self.base_url, path), **kwargs)
        r.raise_for_status()
        if r.status_code==204: return True
        ct = (r.headers.get('content-type') or '').split(';')[0]
        return r.json() if ct=='application/json' else r.text

    def _kpath(self, kernel_id='', suffix=''): return f"/api/kernels/{kernel_id}{suffix}" if kernel_id else '/api/kernels'

    async def kernel_request(self, method, kernel_id='', suffix='', **kwargs):
        return await self._request(method, self._kpath(kernel_id, suffix), **kwargs)

    async def aclose_http(self):
        if self._http and self._own_http and not self._http.is_closed: await self._http.aclose()

## The client

`JupyAsyncKernelClient` is one kernel's worth of everything: the HTTP lifecycle calls for its kernel, the websocket connection, and the messaging API. The HTTP side comes first, since it can run before any websocket exists; then the private machinery that moves frames; then the public messaging surface, each method demonstrated as soon as it is defined.


In [ ]:
#| export
class JupyAsyncKernelClient(EvalOps, KernelApi):
    "AsyncKernelClient-ish API over the kernels HTTP API plus its websocket channels."
    allow_stdin = True
    def __init__(self, base_url, kernel_id=None, token=None, session_id=None, username=None, headers=None, timeout=30, http_client=None,
        reconnect=True, reconnect_ceiling=300.0):
        super().__init__(base_url, token=token, headers=headers, timeout=timeout, http_client=http_client)
        self.kernel_id = kernel_id
        self.owned = False    # True only when `connect` created the kernel; honored by `__aexit__`
        self.session_id = session_id or uuid.uuid4().hex
        self.session = Session(session=self.session_id, username=username or os.environ.get("USER") or "")
        self._ws,self._start_task,self._send_task,self._recv_task,self._close_task = [None]*5
        self.reconnect,self.reconnect_ceiling,self._unsent,self._closing = reconnect,reconnect_ceiling,None,False
        self._send_q = asyncio.Queue()
        self._queues = {k: asyncio.Queue() for k in ("shell", "iopub", "stdin", "control")}
        self._reply_waiters = {k: {} for k in ("shell", "control")}
        self._stale_replies = {k: set() for k in ("shell", "control")}
        self._last_stdin_req = None

    def _kpath(self, kernel_id='', suffix=''): return super()._kpath(kernel_id or self.kernel_id, suffix)

    async def kernel_request(self, method, suffix="", **kwargs):
        if not self.kernel_id: return None
        return await super().kernel_request(method, self.kernel_id, suffix, **kwargs)

In [ ]:
#| export
@patch
async def start_kernel(self:JupyAsyncKernelClient, kernel_name="python3", **kwargs):
    model = await self._request("POST", self._kpath(''), json={"name": kernel_name, **kwargs})
    self.kernel_id = model["id"]
    return model

`start_kernel` posts to `/api/kernels` and binds the returned id. A live server to try it on - jupygate in a background thread - and the rest of the page's client:


In [ ]:
server = serve(create_app(), port=8811, in_thread=True)
for _ in range(100):
    if getattr(server, 'started', False): break
    time.sleep(0.05)
base_url = 'http://127.0.0.1:8811'
server.started


True

In [ ]:
kc = JupyAsyncKernelClient(base_url)
model = await kc.start_kernel()
model

{'id': '33873429da3d4fa38f77460c794d0f33',
 'name': '',
 'execution_state': 'alive',
 'connections': 0}

The remaining HTTP lifecycle calls are one-liners over `kernel_request`; `is_alive` doubles as an existence check since the server 404s unknown kernel ids.

In [ ]:
#| export
@patch
async def shutdown_kernel(self: JupyAsyncKernelClient): return await self.kernel_request("DELETE")
@patch
async def interrupt_kernel(self: JupyAsyncKernelClient): return await self.kernel_request("POST", "/interrupt")
@patch
async def restart_kernel(self: JupyAsyncKernelClient): return await self.kernel_request("POST", "/restart")

@patch
async def is_alive(self: JupyAsyncKernelClient):
    try: return bool(await self.kernel_request("GET"))
    except Exception: return False


In [ ]:
test_eq(await kc.is_alive(), True)
none_kc = JupyAsyncKernelClient(base_url)
test_eq(await none_kc.is_alive(), False)

### How it works

Every inbound frame lands in `_route_reply_or_queue`, which enforces the routing rules: a message whose `parent_header.msg_id` has a waiting `reply=True` future resolves it (and with `fail_pending`, an error or abort fails every other pending future immediately); everything else queues on its channel for the `get_*_msg` accessors. It also remembers the last `input_request` header, which `input()` uses below.

In [ ]:
#| export
@patch
def _fail_pending(self: JupyAsyncKernelClient, exc, channel="shell", skip=None):
    "Fail all pending reply waiters on `channel` (except `skip`), e.g. when an error aborts the kernel's queue."
    for mid, (fut, _) in list(self._reply_waiters[channel].items()):
        if mid != skip and not fut.done(): fut.set_exception(exc)

@patch
def _route_reply_or_queue(self: JupyAsyncKernelClient, msg):
    channel = msg.pop("channel", None) or "shell"
    msg.setdefault("msg_id", msg.get("header", {}).get("msg_id"))
    msg.setdefault("msg_type", msg.get("header", {}).get("msg_type"))
    msg.setdefault("buffers", [])
    if channel == "stdin" and msg["msg_type"] == "input_request": self._last_stdin_req = msg.get("header")
    parent_msg_id = msg.get("parent_header", {}).get("msg_id")
    if channel in self._reply_waiters and parent_msg_id:
        pend = self._reply_waiters[channel].pop(parent_msg_id, None)
        if pend:
            fut, fail_pending = pend
            if not fut.done(): fut.set_result(msg)
            cts = msg.get("content", {})
            if fail_pending and cts.get("status") in ("error", "aborted"):
                exc = RuntimeError(f"Kernel error aborted: {cts.get('ename')}: {cts.get('evalue')}")
                self._fail_pending(exc, channel, skip=parent_msg_id)
            return
        if parent_msg_id in self._stale_replies[channel]:
            self._stale_replies[channel].discard(parent_msg_id)
            return
    q = self._queues.get(channel)
    if q: q.put_nowait(msg)

In [ ]:
#| export
@patch
async def _ensure_started(self: JupyAsyncKernelClient):
    if not self._start_task: self.start_channels()
    if self._start_task: await self._start_task

@patch
def _queue_msg(self: JupyAsyncKernelClient, msg, channel: str):
    msg = dict(msg)
    msg["channel"] = channel
    bufs = msg.get("buffers")
    if bufs:
        msg["buffers"] = [bytes(b) for b in bufs]
        payload = serialize_binary_message(msg)
    else:
        msg.pop("buffers", None)
        payload = dumps(msg)
    self._send_q.put_nowait(payload)
    return msg["header"]["msg_id"]

@patch
async def _get_msg(self: JupyAsyncKernelClient, channel, timeout=None):
    await self._ensure_started()
    q = self._queues[channel]
    try:
        async with asyncio.timeout(timeout): return await q.get()
    except asyncio.TimeoutError as e: raise Empty from e

In [ ]:
#| export
@patch
async def _send_loop(self: JupyAsyncKernelClient):
    assert self._ws is not None
    while True:
        payload = self._unsent if self._unsent is not None else await self._send_q.get()
        if payload is None: return
        self._unsent = payload
        try: await self._ws.send(payload)
        except Exception as e:
            log.warning("websocket send failed: %s", e)
            return  # the payload stays in `_unsent`: a reconnected send loop retries it first
        self._unsent = None

@patch
async def _recv_loop(self: JupyAsyncKernelClient):
    assert self._ws is not None
    with suppress(websockets.ConnectionClosed):
        async for data in self._ws:
            if isinstance(data, str): msg = loads(data)
            elif isinstance(data, bytes): msg = deserialize_binary_message(data)
            else: continue
            self._route_reply_or_queue(msg)
    if self.reconnect and not self._closing: self._start_task = asyncio.create_task(self._reconnect())

In [ ]:
#| export
@patch
async def _start_ws(self: JupyAsyncKernelClient):
    if self._ws and self._ws.close_code is None: return
    for t in (self._send_task, self._recv_task):
        if t and not t.done(): t.cancel()  # a stale send loop must not steal payloads from the new one
    params = {"session_id": self.session_id}
    if self.token: params["token"] = self.token
    ws_url = _join_url(self.base_url, self._kpath(suffix="/channels"), ws=True, params=params)
    self._ws = await websockets.connect(ws_url, additional_headers=self._headers, ping_interval=30)
    self._send_task = asyncio.create_task(self._send_loop())
    self._recv_task = asyncio.create_task(self._recv_loop())

Sending is the mirror image: `_exec_req` builds a signed message, queues its frame, and either returns the msg_id or awaits the reply future. `__getattr__` then makes *every* `*_request` message type in the protocol callable by name - the typed wrappers further down only add convenient defaults. `wait_for_ready` is one `kernel_info` round trip plus an iopub drain, which over a websocket is purely about the kernel being up (there is no subscription race to compensate for - the gateway handled that on the zmq side).

In [ ]:
#| export
@patch
async def _send_and_await_reply(self: JupyAsyncKernelClient, msg, msg_id, timeout=None, channel="shell", fail_pending=False):
    fut = asyncio.get_running_loop().create_future()
    self._reply_waiters[channel][msg_id] = (fut, fail_pending)
    self._queue_msg(msg, channel)
    try:
        async with asyncio.timeout(timeout): return await fut
    finally:
        popped = self._reply_waiters[channel].pop(msg_id, None)
        if popped and popped[0] is fut and (not fut.done() or fut.cancelled()): self._stale_replies[channel].add(msg_id)

In [ ]:
#| export
@patch
def _exec_req(self: JupyAsyncKernelClient, name, content=None, channel="shell", metadata=None, reply=False, timeout=None, subshell_id=None,
    parent=None, fail_pending=False):
    msg = self.session.msg(name, content, metadata=metadata, parent=parent)
    if subshell_id: msg["header"]["subshell_id"] = subshell_id
    msg_id = msg["header"]["msg_id"]
    if not reply: return self._queue_msg(msg, channel)
    return self._send_and_await_reply(msg, msg_id, timeout=timeout, channel=channel, fail_pending=fail_pending)

def _gen_request(self, name):
    "Generated `*_request`/`*_reply` senders; other names raise, so typos fail instead of sending bogus messages. Assigned onto the class below (a module-level `__getattr__` would become a PEP 562 hook)."
    if name.startswith("_") or not name.endswith(("_request", "_reply")): raise AttributeError(name)
    def _f(reply=False, timeout=None, channel="shell", metadata=None, subshell_id=None, parent=None, fail_pending=False, **kwargs):
        return self._exec_req(name, content=kwargs or None, reply=reply, timeout=timeout, channel=channel, metadata=metadata,
            subshell_id=subshell_id, parent=parent, fail_pending=fail_pending)
    return _f

JupyAsyncKernelClient.__getattr__ = _gen_request

### Connecting

In [ ]:
#| export
@patch
def start_channels(self: JupyAsyncKernelClient, shell=True, iopub=True, stdin=True, control=True):
    if not (shell or iopub or stdin or control): return
    if self._start_task and not self._start_task.done(): return
    self._start_task = asyncio.create_task(self._start_ws())
    return self

In [ ]:
#| export
@patch(as_prop=True)
def channels_running(self: JupyAsyncKernelClient): return bool(self._ws and self._ws.close_code is None)

`start_channels` opens the websocket (session id and token as query params) and starts the loops; `channels_running` reports the live state. It returns immediately: readiness is `wait_for_ready`'s job.


The `get_*_msg` accessors read each channel's queue, `jupyter_client`-style, raising `queue.Empty` on timeout. With a request sent fire-and-forget (no `reply=True`), the reply simply arrives on the shell queue - demonstrated just below, once connected.

In [ ]:
#| export
@patch
async def get_shell_msg(self: JupyAsyncKernelClient, timeout=None): return await self._get_msg("shell", timeout)
@patch
async def get_iopub_msg(self: JupyAsyncKernelClient, timeout=None): return await self._get_msg("iopub", timeout)
@patch
async def get_stdin_msg(self: JupyAsyncKernelClient, timeout=None): return await self._get_msg("stdin", timeout)
@patch
async def get_control_msg(self: JupyAsyncKernelClient, timeout=None): return await self._get_msg("control", timeout)

In [ ]:
#| export
@patch
async def wait_for_ready(self: JupyAsyncKernelClient, timeout=None):
    await self._ensure_started()
    await self.kernel_info_request(reply=True, timeout=timeout)
    while True:
        try: await self.get_iopub_msg(timeout=0.05)
        except Empty: return

`wait_for_ready` is one `kernel_info` round trip plus an iopub drain. Over a websocket that is purely about the kernel being up: there is no subscription race to compensate for, since the gateway handled zmq subscription on its side. It calls the generated `kernel_info_request` rather than the typed wrapper defined further down, so the machinery above is all it needs.


In [ ]:
kc.start_channels()
await kc.wait_for_ready(timeout=60)
kc.channels_running


True

A fire-and-forget request's reply, read off the queue (this also shows `_gen_request` at work: `comm_info_request` has no typed wrapper here):

In [ ]:
kc.comm_info_request()
m = await kc.get_shell_msg(timeout=15)
m['msg_type']

'comm_info_reply'

In [ ]:
#| export
_replkw = dict(reply=False, timeout=None)
_subskw = dict(**_replkw, subshell_id=None)

### The request API

In [ ]:
#| export
@patch
@use_kwargs_dict(**_subskw)
def execute(self: JupyAsyncKernelClient, code, silent=False, store_history=True, user_expressions=None, allow_stdin=None, stop_on_error=True,
    fail_pending=False, **kwargs):
    user_expressions = {} if user_expressions is None else user_expressions
    allow_stdin = self.allow_stdin if allow_stdin is None else allow_stdin
    if not isinstance(code, str): raise ValueError(f"code {code!r} must be a string")
    validate_string_dict(user_expressions)
    return self.execute_request(code=code, silent=silent, store_history=store_history, user_expressions=user_expressions,
        allow_stdin=allow_stdin, stop_on_error=stop_on_error, fail_pending=fail_pending, **kwargs)

`execute` with `reply=True` awaits the `execute_reply`; iopub arrives independently on its queue.

In [ ]:
rep = await kc.execute("print('hello'); 6*7", reply=True, timeout=30)
test_eq(rep['content']['status'], 'ok')
outs = {}
while len(outs) < 2:
    m = await kc.get_iopub_msg(timeout=15)
    if m['msg_type']=='stream': outs['stream'] = m['content']['text'].strip()
    if m['msg_type']=='execute_result': outs['result'] = m['content']['data']['text/plain']
outs

{'stream': 'hello', 'result': '42'}

Requests are concurrency-safe: fire several, gather the replies; each future resolves from its own parent msg_id.

In [ ]:
reps = await asyncio.gather(*[kc.execute(f'{i}*{i}', reply=True, timeout=30) for i in range(5)])
[r['content']['status'] for r in reps]

['ok', 'ok', 'ok', 'ok', 'ok']

`fail_pending=True` turns a queue-aborting error into immediate failures for everything still pending, instead of a drip of `aborted` replies.

In [ ]:
bad = kc.execute('1/0', reply=True, timeout=30, stop_on_error=True, fail_pending=True)
rest = [kc.execute('2+2', reply=True, timeout=30) for _ in range(3)]
r0 = await bad
test_eq(r0['content']['status'], 'error')
failed = 0
for t in rest:
    try: await t
    except RuntimeError: failed += 1
failed

0

Subshells (JEP 91) pass through gateway and kernel untouched: create one on the control channel, run a cell on it concurrently with the main shell.

In [ ]:
#| export
@patch
@use_kwargs_dict(**_replkw)
def create_subshell(self: JupyAsyncKernelClient, **kwargs): return self.create_subshell_request(channel="control", **kwargs)

@patch
@use_kwargs_dict(**_replkw)
def list_subshell(self: JupyAsyncKernelClient, **kwargs): return self.list_subshell_request(channel="control", **kwargs)

@patch
@use_kwargs_dict(**_replkw)
def delete_subshell_request(self: JupyAsyncKernelClient, subshell_id: str, metadata=None, **kwargs):
    return self._exec_req("delete_subshell_request", content={"subshell_id": subshell_id}, channel="control", metadata=metadata, **kwargs)

@patch
@use_kwargs_dict(**_replkw)
def delete_subshell(self: JupyAsyncKernelClient, subshell_id: str, **kwargs): return self.delete_subshell_request(subshell_id=subshell_id, **kwargs)

In [ ]:
sub = (await kc.create_subshell(reply=True, timeout=15))['content']['subshell_id']
rep = await kc.execute('40+2', reply=True, timeout=30, subshell_id=sub)
await kc.delete_subshell(sub, reply=True, timeout=15)
rep['content']['status'], rep['header'].get('subshell_id') is not None

('ok', False)

`input()` answers the pending prompt, parented to it (the router remembered the `input_request` header).

In [ ]:
#| export
@patch
def input(self: JupyAsyncKernelClient, string: str):
    "Answer an `input_request`, parented to it so kernels and gateways can match reply to prompt."
    parent, self._last_stdin_req = self._last_stdin_req, None
    self.input_reply(value=string, channel="stdin", parent=parent)

In [ ]:
fut = asyncio.ensure_future(kc.execute("name = input('who? ')", reply=True, timeout=30))
prompt = await kc.get_stdin_msg(timeout=15)
kc.input('Jeremy')
await fut
rep = await kc.execute('name', reply=True, timeout=30, user_expressions={'v':'name'})
rep['content']['user_expressions']['v']['data']['text/plain']

"'Jeremy'"

The raw `*_request` senders mirror the wire protocol (fire-and-forget, or `reply=True` for the full reply message). On top of them, `complete`, `inspect`, and `check` are the ergonomic verbs a frontend actually wants — unwrapped values, cursor position defaulted to the end:


In [ ]:
#| export
@patch
async def complete(self:JupyAsyncKernelClient, code, cursor_pos=None, timeout=15):
    "Completion `(matches, replace_start)` for `code` at `cursor_pos` (end when None)"
    cursor_pos = len(code) if cursor_pos is None else cursor_pos
    c = (await self.complete_request(code=code, cursor_pos=cursor_pos, reply=True, timeout=timeout))['content']
    return c.get('matches', []), c.get('cursor_start', cursor_pos)


In [ ]:
matches, start = await kc.complete('imp')
assert 'import' in matches
matches[:3]


['import']

In [ ]:
#| export
@patch
async def inspect(self:JupyAsyncKernelClient, code, cursor_pos=None, detail_level=0, timeout=15):
    "Inspection text for `code` at `cursor_pos` ('' when nothing found)"
    cursor_pos = len(code) if cursor_pos is None else cursor_pos
    c = (await self.inspect_request(code=code, cursor_pos=cursor_pos, detail_level=detail_level, reply=True, timeout=timeout))['content']
    return c.get('data', {}).get('text/plain', '') if c.get('found') else ''

@patch
async def check(self:JupyAsyncKernelClient, code, timeout=15):
    "('complete'|'incomplete'|'invalid'|'unknown', indent) for `code` as a cell"
    c = (await self.is_complete_request(code=code, reply=True, timeout=timeout))['content']
    return c.get('status', 'unknown'), c.get('indent', '')


In [ ]:
txt = await kc.inspect('print')
assert 'print' in txt
test_eq(await kc.inspect('no_such_name_here'), '')


True

In [ ]:
#| export
@patch
@use_kwargs_dict(**_subskw, keep=True)
def history(self: JupyAsyncKernelClient, raw=True, output=False, hist_access_type="range", **kwargs):
    if hist_access_type=="range":
        kwargs.setdefault("session", 0)
        kwargs.setdefault("start", 0)
    return self.history_request(raw=raw, output=output, hist_access_type=hist_access_type, **kwargs)

In [ ]:
h = await kc.history(reply=True, timeout=15)
len(h['content']['history']) > 0

True

`comm_info` and `kernel_info` keep the conventional wrapper shape; `check` (over `is_complete_request`) is the one frontends poll while the user types:


In [ ]:
#| export
@patch
@use_kwargs_dict(**_subskw)
def comm_info(self: JupyAsyncKernelClient, target_name=None, **kwargs): return self.comm_info_request(target_name=target_name, **kwargs)

@patch
@use_kwargs_dict(**_subskw)
def kernel_info(self: JupyAsyncKernelClient, **kwargs): return self.kernel_info_request(**kwargs)

In [ ]:
test_eq(await kc.check('for i in range(3):'), ('incomplete', '    '))
test_eq((await kc.check('1+1'))[0], 'complete')


'incomplete'

### Calling kernel functions

`reply` is sugar for `execute(reply=True)`. `eval` turns the `user_expressions` round trip into a function call: run `func(*args, **kw)` kernel-side (awaiting coroutines), bring the result back by repr, and reconstruct it client-side — `try_eval` wraps primitive results in a dynamic class named after the kernel-side type. `_call=False` evaluates `func` as a bare expression. `ipy` targets `get_ipython()` methods, and the generated service methods mirror what `ipyfuncs` patches onto the kernel's shell (`sig_help`, `get_schemas`, `ranked_complete`, ...). The whole family is inherited from jupywire's `EvalOps` mixin over this module's `reply` — one definition shared with `conkernelclient`, so callers read identically over zmq and websockets. `priority=` routes to a dedicated subshell; only hosts that create one set `self.priority`, and the funnel asserts otherwise.

In [ ]:
#| export
@patch
def reply(self:JupyAsyncKernelClient, code, user_expressions=None, timeout=None, priority=False, **kw):
    "Sugar for `execute(reply=True)`: run `code` and await its `execute_reply`"
    if priority: assert self.priority, 'no priority subshell configured'
    return self.execute(code, user_expressions=user_expressions, reply=True, timeout=timeout,
        subshell_id=self.priority if priority else None, **kw)


In [ ]:
r = await kc.reply('def add(a, b): return a+b')
test_eq(r['content']['status'], 'ok')
test_eq(await kc.eval('add', a=10, b=20), 30)
await kc.reply('a = [1,2,3]')
test_eq(await kc.eval('a', _call=False), [1,2,3])
test_eq(await kc.eval('add', a=30, b=40, _literal=False), '70')
_r = await kc.eval('missing_fn')
assert 'NameError' in _r

The `_ipy_funcs` services live kernel-side in [`ipyfuncs`](https://github.com/AnswerDotAI/ipyfuncs); importing it is the whole setup, so the battery is self-sufficient:

In [ ]:
await kc.reply('import ipyfuncs')
await kc.reply('''def range_ex(
    a:str  # some param
):
    "some func docstring"
    ...''')
sigs = await kc.sig_help(code='range_ex(', line_no=1, col_no=9)
test_eq(sigs[0]['label'], 'range_ex')
schemas = await kc.get_schemas(fs=['range_ex'])
test_eq(schemas['range_ex']['function']['name'], 'range_ex')
await kc.xpush(asdf=4)
test_eq(await kc.retr('asdf'), 4)
test_eq(await kc.eval_exprs(vs=['list(range(5))']), {'list(range(5))': [0,1,2,3,4]})
await kc.xenv(hi='jupy')
test_eq(await kc.eval('__os.environ["hi"]', _call=False), 'jupy')

### Streaming an execution: `run`

`execute` returns when the request is sent (or, with `reply=True`, when the reply lands); iopub arrives independently, any time, interleaved with every other client's traffic. That contract is right for the raw protocol and wrong for a REPL-shaped consumer, which wants "*this* execution's outputs, in order, until it finishes". `Run` owns that view: it filters iopub by parent msg id (out-of-band traffic — another client's outputs, a stray idle from a silent exec — never leaks in), converts each output message to its nbformat shape via `msg2out`, and ends only when both the `execute_reply` and this execution's own `idle` have arrived. `input_request`s parented to the run are answered through the `on_stdin` callback (no callback means `allow_stdin=False`, so kernel-side `input()` fails fast in-band rather than hanging); comm messages — deliberately unfiltered, they are kernel-adjacent state, not execution output — pass to `on_comm` when given. On ~1s of iopub silence the kernel is polled over HTTP, so a dead kernel raises `DeadKernelError` instead of hanging the iterator. A `Run` is single-use: iterate it once, or `await` it to drain into a plain output list (the LLM-shaped collected mode is the streamed one plus one line).

In [ ]:
#| export
OUTPUT_MSGS = ('stream', 'display_data', 'execute_result', 'error')
COMM_MSGS = ('comm_open', 'comm_msg', 'comm_close')

class DeadKernelError(RuntimeError): pass

class Run:
    "One execution's typed output stream: parent-filtered iopub until reply+idle; single-use."
    def __init__(self, kc, code, on_stdin=None, on_comm=None, **kw):
        self.kc,self.on_stdin,self.on_comm = kc,on_stdin,on_comm
        self.msg_id = kc.execute(code, allow_stdin=on_stdin is not None, **kw)
        self.reply,self.status,self.execution_count = None,None,None

    async def __aiter__(self):
        chans = dict(iopub=self.kc.get_iopub_msg, shell=self.kc.get_shell_msg, stdin=self.kc.get_stdin_msg)
        pend = {ch: asyncio.ensure_future(f(timeout=None)) for ch,f in chans.items()}
        done = idle = False
        try:
            while not (done and idle):
                ready,_ = await asyncio.wait(pend.values(), return_when=asyncio.FIRST_COMPLETED, timeout=1)
                if not ready:
                    if not await self.kc.is_alive(): raise DeadKernelError('kernel died while executing')
                    continue
                for t in ready:
                    ch = next(k for k,v in pend.items() if v is t)
                    msg = t.result()
                    pend[ch] = asyncio.ensure_future(chans[ch](timeout=None))
                    mt,c = msg['msg_type'],msg['content']
                    mine = nested_idx(msg, 'parent_header', 'msg_id') == self.msg_id
                    if ch == 'shell':
                        if mt == 'execute_reply' and mine:
                            self.reply,self.status,self.execution_count = msg,c.get('status'),c.get('execution_count')
                            done = True
                    elif ch == 'stdin':
                        if mt == 'input_request' and mine: self.kc.input(await self.on_stdin(c.get('prompt', ''), c.get('password', False)))
                    elif mt == 'status' and c.get('execution_state') == 'idle' and mine: idle = True
                    elif mt in OUTPUT_MSGS and mine: yield msg2out(msg)
                    elif mt in COMM_MSGS and self.on_comm is not None: self.on_comm(mt, c)
        finally:
            for t in pend.values(): t.cancel()

    async def collect(self):
        "Drain the run, returning all outputs as a list."
        return [o async for o in self]

    def __await__(self): return self.collect().__await__()

@patch
def run(self:JupyAsyncKernelClient, code, on_stdin=None, on_comm=None, **kw):
    "Execute `code`, returning a `Run` over this execution's nbformat-shaped outputs."
    return Run(self, code, on_stdin=on_stdin, on_comm=on_comm, **kw)

The REPL view, streamed: outputs arrive as nbformat dicts while the cell runs, and the reply's verdict lands on the `Run` when iteration ends:

In [ ]:
run = kc.run("print('hi'); 6*7")
outs = [o async for o in run]
test_eq([o['output_type'] for o in outs], ['stream', 'execute_result'])
test_eq(run.status, 'ok')
assert run.execution_count
outs

Awaiting the run drains it — the collected mode LLM-shaped clients want (e.g. clikernel's `render_outs(await kc.run(code))`). An error is an ordinary output plus the reply's verdict, so a consumer chooses its own severity:

In [ ]:
bad = kc.run('1/0')
outs = await bad
test_eq(outs[0]['output_type'], 'error')
test_eq(outs[0]['ename'], 'ZeroDivisionError')
test_eq(bad.status, 'error')


`on_stdin` answers this run's `input_request`s (an async callable, prompt and password flag in, reply text out). Without it, `allow_stdin` is off and kernel-side `input()` fails fast instead of hanging — the right default for unattended clients:

In [ ]:
async def whoami(prompt, password): return 'jupy'
run = kc.run("nm = input('who? ')", on_stdin=whoami)
await run
test_eq(run.status, 'ok')
test_eq(await kc.retr('nm'), 'jupy')

`on_comm` receives comm traffic as `(msg_type, content)`. Comms are how a kernel-side magic talks to its host app (e.g. ipyai's `%ipyai`), so they are deliberately *not* parent-filtered — kernel-adjacent state, not execution output — and never appear in the output stream:

In [ ]:
comms = []
run = kc.run("from comm import create_comm\ncm = create_comm('demo', data=dict(x=1))\ncm.send(dict(y=2))",
    on_comm=lambda mt,c: comms.append((mt, c.get('data'))))
outs = await run
test_eq(outs, [])                       # comm traffic is not output
test_eq(comms[0], ('comm_open', dict(x=1)))
test_eq(comms[1], ('comm_msg', dict(y=2)))

The filtering contract, live: iopub is a shared broadcast, so an out-of-band execute (a host's silent bridge exec, another client) broadcasts its own outputs and its own `idle` while a run is collecting. Neither may leak in — a foreign output must not appear in the stream, and a foreign idle must not end the run early:

In [ ]:
kc.execute("'FOREIGN'")                 # fire-and-forget: its result and idle hit iopub first
outs = await kc.run("'MINE'")
test_eq(len(outs), 1)
assert 'MINE' in outs[0]['data']['text/plain'] and 'FOREIGN' not in str(outs)

### Reconnecting

Over zmq a dropped connection is invisible: the transport redials by itself and jupyter_client never knows. A websocket client has to earn that property. When the receive loop ends without `aclose` having been called, the client redials `/channels` with the same `session_id`, which is what lets a gateway hand back the surviving queue and replay what was missed (see jupygate's reconnect section); the send loop resumes behind it, and a frame that died mid-send is resent first. Pending `reply=True` futures deliberately survive the drop: the gateway replays the very replies they await, so failing them would be premature.

Giving up is a separate decision from retrying. Each failed redial probes the kernel over HTTP: if the server answers and the kernel is gone, retrying would be a lie, so pending futures fail immediately with the reason. If the server itself is unreachable, the client retries with backoff until `reconnect_ceiling` seconds have passed, then fails them with `ConnectionError`. `reconnect=False` restores fail-fast behavior.

In [ ]:
#| export
@patch
async def _reconnect(self: JupyAsyncKernelClient):
    "Redial with the same session id, with backoff; on a dead kernel or an expired ceiling, fail the pending futures."
    deadline, delay = time.monotonic() + self.reconnect_ceiling, 0.1
    while not self._closing:
        try:
            await self._start_ws()
            log.info("websocket reconnected")
            return
        except Exception as e:
            exc = None
            try: await self.kernel_request('GET')
            except httpx.HTTPStatusError as he: exc = RuntimeError(f'kernel {self.kernel_id} is gone: {he}')
            except Exception: pass  # the server is unreachable too: keep trying until the ceiling
            if exc is None and time.monotonic() > deadline: exc = ConnectionError(f'gave up reconnecting after {self.reconnect_ceiling}s: {e}')
            if exc:
                for c in ('shell','control'): self._fail_pending(exc, c)
                raise exc
            await asyncio.sleep(delay)
            delay = min(delay*2, 5.0)

In [ ]:
fut = asyncio.ensure_future(kc.execute("import time; time.sleep(1); 'survived'", reply=True, timeout=20))
await asyncio.sleep(0.3)
kc._ws.transport.abort()  # the network dies mid-cell; a deliberate close would not redial
rep = await fut
test_eq(rep['content']['status'], 'ok')
test_eq((await kc.execute('1+1', reply=True, timeout=15))['content']['status'], 'ok')
kc.channels_running

True

### Closing down

`shutdown` goes over the control channel; the HTTP `DELETE` (via `shutdown_kernel`) is the more common route since it also reaps the server-side process.

In [ ]:
#| export
@patch
@use_kwargs_dict(**_replkw)
def shutdown(self: JupyAsyncKernelClient, restart=False, **kwargs):
    return self.shutdown_request(restart=restart, channel="control", **kwargs)

`aclose` unwinds everything the client owns - loops, websocket, pending futures, and the HTTP client if it owns it; `stop_channels` is its fire-and-forget form for sync contexts. Shutting the kernel down first makes the teardown complete:

In [ ]:
#| export
@patch
async def aclose(self: JupyAsyncKernelClient):
    self._closing = True
    for t in (self._start_task, self._send_task, self._recv_task):
        if t and not t.done(): t.cancel()
    if self._ws and self._ws.close_code is None: await self._ws.close()
    for t in (self._start_task, self._send_task, self._recv_task):
        if t:
            with suppress(asyncio.CancelledError, Exception): await t
    for d in self._reply_waiters.values():
        for fut, _ in d.values():
            if not fut.done(): fut.cancel()
        d.clear()
    for s in self._stale_replies.values(): s.clear()
    self._ws = None
    await self.aclose_http()

@patch
def stop_channels(self: JupyAsyncKernelClient):
    if self._close_task and not self._close_task.done(): return
    self._close_task = asyncio.create_task(self.aclose())

### The packaged startup

Construct, create, open channels, wait ready is the four-line dance every consumer types; `connect` packages it, covering attach with the same verb (`kernel=` an existing id). Ownership is recorded where it's decided: a kernel `connect` *created* is `owned`, and the context manager honors that on the way out — the client always closes, but the kernel is shut down only when owned. An attached kernel is never stopped, and bare `aclose` never kills anything: explicit `shutdown_kernel` remains the only other way a kernel ends.

In [ ]:
#| export
@patch(cls_method=True)
async def connect(cls:JupyAsyncKernelClient, base_url, kernel=None, token=None, timeout=60, **kw):
    "Construct + create a kernel (or attach to `kernel`) + open channels + wait ready; a created kernel is `owned`"
    self = cls(base_url, kernel_id=kernel, token=token)
    if kernel is None:
        await self.start_kernel(**kw)
        self.owned = True
    self.start_channels()
    await self.wait_for_ready(timeout=timeout)
    return self

@patch
async def __aenter__(self:JupyAsyncKernelClient): return self

@patch
async def __aexit__(self:JupyAsyncKernelClient, *exc):
    if self.owned:
        with suppress(Exception): await self.shutdown_kernel()
    await self.aclose()

In [ ]:
async with await JupyAsyncKernelClient.connect(server.url) as k2:
    kid2 = k2.kernel_id
    assert k2.owned and k2.channels_running
assert not await JupyAsyncKernelClient(server.url, kernel_id=kid2).is_alive()   # owned: died with the block

Attaching with the same verb: `kernel=` binds an existing id, `owned` stays False, and leaving the block leaves the kernel running:

In [ ]:
async with await JupyAsyncKernelClient.connect(server.url, kernel=kc.kernel_id) as att: assert not att.owned
test_eq(await kc.is_alive(), True)   # attached: the block leaves it running

In [ ]:
await kc.shutdown_kernel()
await kc.aclose()
test_eq(kc.channels_running, False)
test_eq(await none_kc.is_alive(), False)
await none_kc.aclose()


In [ ]:
#| hide
server.should_exit = True

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()